In [1]:
from brian2 import *
import sys
# sys.path = [p for p in sys.path if 'Neuron and Synapse Models' not in p and 'Tools' not in p]
sys.path.append('Neuron and Synapse Models')
sys.path.append('Tools')

from neuronModels import *
from ringAttractorTEMP import *
from plottingTools import *

import matplotlib.pyplot as plt

# Simulation parameters
defaultclock.dt = 0.1*ms

AttributeError: 'Quantity' object has no attribute 'iscompound'

In [ ]:
# Convert slider values to Brian units
tau = 10 * ms
sigma_noise = 1 * mV
V_rest = -70 * mV
I0 = 30 * mV
g_exc = 0.1* mV
g_inh = -0.15 * mV
num_neurons = 120

stimulus_center=0 #rad
stimulus_width=0.5


# Define neuron positions
positions = linspace(0, 2*pi, num_neurons, endpoint=False)

# Calculate external input
d = np.angle(np.exp(1j * (positions - stimulus_center)))
I_ext_array = I0 * np.exp(-(d**2) / (2 * stimulus_width**2))


# Set up neuron model
neuron_eq = Equations(LIF_xi_eq, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise)

# Set up ring attractor
Vth = -48 * mV
V_reset = -80 * mV
refractory_period = 5 * ms

# Create the ring attractor network
ringAttractor = RingAttractor(neuron_eq, 
                        num_neurons, 
                        Vth, V_reset, refractory_period,
                        syn_profile=syn_profile,
                        autapse=True,
                        sigma_exc=sigma_exc_val, 
                        sigma_inh=sigma_inh_val, 
                        g_exc=g_exc, 
                        g_inh=g_inh) 

# Set external input
ringAttractor.ring_pool.I_ext = I_ext_array

# Setup monitors
spikemon = SpikeMonitor(ringAttractor.ring_pool)
statemon = StateMonitor(ringAttractor.ring_pool, 'V', record=True)
inputmon = StateMonitor(ringAttractor.ring_pool, 'I_ext', record=True)

net = Network(ringAttractor.BrianObjects + [spikemon, statemon, inputmon])

input_on = 2*second
input_off = 2*second
sim_duration=input_on+input_off

# Run simulation
net.run(input_on)
# Turn off input for the second half
ringAttractor.ring_pool.I_ext = I_ext_array * 0
net.run(input_off)

firing_rate, _ = firing_rate_profile(spikemon, positions/(2*pi), sim_duration)

In [ ]:
sys.path.pop()
sys.path.pop()